# Getting Started With LangChain And Open AI

In this quickstart we'll see how to:

- Get setup with LangChain, LangSmith and LangServe
- Use the most basic and common components of LangChain: prompt templates, models, and output parsers.
- Build a simple application with LangChain
- Trace your application with LangSmith
- Serve your application with LangServe

In [10]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate, 
    HumanMessagePromptTemplate
)

In [2]:
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

## LangSmith for tracking our workflow
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]=os.getenv("LANGCHAIN_TRACING_V2")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGSMITH_ENDPOINT"]=os.getenv("LANGSMITH_ENDPOINT")


print("GROQ:", bool(os.getenv("GROQ_API_KEY")))
print("LANGSMITH:", bool(os.getenv("LANGSMITH_API_KEY")))
print("PROJECT:", os.getenv("LANGCHAIN_PROJECT"))
print("ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))

GROQ: True
LANGSMITH: True
PROJECT: GEN_AI_WORKFLOW
ENDPOINT: https://api.smith.langchain.com


In [ ]:
modelo = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.6
)

# usando o tracking do langSmith, na primeira execução, deu erro por que o modelo que eu tinha configurado de inicio, não estava mais disponivel
# procurei por quais modelos a minha api_key fornecia, depois da atualização que tiveram, e esse llm gpt-oss parece valer a pena pro meu uso aqui 

# total de tokens gastos nessa execução -> 136 e levou 0.55 segundos até run total
modelo.invoke('Teste')

AIMessage(content='Olá! Como posso ajudar?', additional_kwargs={'reasoning_content': 'The user just says "Teste". Likely they are testing. Should respond appropriately, maybe ask how can I help. Use Portuguese? The user wrote "Teste". Could be Portuguese. Respond in Portuguese: "Olá! Como posso ajudar?"'}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 72, 'total_tokens': 136, 'completion_time': 0.136746438, 'completion_tokens_details': {'reasoning_tokens': 49}, 'prompt_time': 0.00324913, 'prompt_tokens_details': None, 'queue_time': 0.270961551, 'total_time': 0.139995568}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05cf2-23f2-7230-ab52-7a8459db699c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 64, 'total_tokens': 136, 'output_token_details': {'reasoning': 49}})

In [ ]:
resposta = modelo.invoke("Quais são as principais criptos no ano de 2026")
print("=== RESPOSTA ===")
print(resposta.content)

print("\n=== MODELO ===")
print(resposta.response_metadata.get("model_name"))

print("\n=== TOKEN USAGE ===")
token_usage = resposta.response_metadata.get("token_usage", {})

# input tokens -> total de tokens enviados (minha pergunta)
# output tokens -> total de tokens que o modelo devolvou, ou seja, a resposta
# input tokens + output = total de tokens que foi contabilizado, cada modelo tem regras de custos diferentes
print(f"Input tokens:  {token_usage.get('prompt_tokens')}")
print(f"Output tokens: {token_usage.get('completion_tokens')}")
print(f"Total tokens:  {token_usage.get('total_tokens')}")

print("\n=== REASONING ===")
print(f"Reasoning tokens: {token_usage.get('completion_tokens_details', {}).get('reasoning_tokens')}")

print("\n=== EXECUÇÃO ===")
print(f"Finish reason: {resposta.response_metadata.get('finish_reason')}")
print(f"Provider:      {resposta.response_metadata.get('model_provider')}")

print("\n=== ID ===")
print(resposta.id)

=== RESPOSTA ===
## Principais criptomoedas em 2026  

A lista abaixo reúne as criptomoedas que, até setembro 2026, se destacam em termos de **capitalização de mercado**, **adaptação institucional**, **atividade de desenvolvedores** e **uso real** (pagamentos, finanças descentralizadas, NFTs, Web 3, etc.). Os rankings podem variar ligeiramente ao longo do ano, mas os projetos abaixo mantêm posições de liderança de forma consistente.

| Rank (aprox.) | Criptomoeda | Ticker | Capitalização de mercado (≈) | Principais usos / diferenciais | Ecossistema / Layer |
|---------------|------------|--------|------------------------------|--------------------------------|----------------------|
| 1 | **Bitcoin** | BTC | US$ 1,3 trilhão | Reserva de valor “digital gold”, meio de pagamento descentralizado, hedge contra inflação. | Base layer (Proof‑of‑Work) |
| 2 | **Ethereum** | ETH | US$ 560 bilhões | Plataforma líder para contratos inteligentes, DeFi, NFTs, dApps. Transição completa para **Proof‑

# 🔍 Análise de Comportamento do Modelo: Alucinação vs. Transparência de Cutoff

### **Objetivo da Análise**
Avaliar a consistência do modelo ao lidar com perguntas sobre dados do futuro (ano de 2026) que ultrapassam seu limite de conhecimento (*knowledge cutoff*).

---

### 📌 **1ª Execução — Resposta Precavida (Transparente)**

* **Pergunta do Usuário:**  
  > *"Quais são as principais criptos no ano de 2026"*

* **Raciocínio Interno (*Reasoning*):**  
  > *"The user asks in Portuguese: 'Quais são as principais criptos no ano de 2026' [...] The model's knowledge cutoff is 2024-06, so we don't have actual 2026 data. We must comply with policy: we can provide a speculative answer, stating that we can give a projection based on trends up to 2024, but we must be clear it's speculative [...]"*

* **Comportamento e Resposta:**  
  O modelo reconheceu explicitamente o seu *knowledge cutoff* (junho de 2024) e iniciou a resposta informando que não possuía dados em tempo real, fornecendo uma projeção especulativa.
  
  > **Aviso do Modelo:** *"Meu treinamento inclui informações até junho de 2024. Não tenho acesso a dados reais de 2026, portanto a lista abaixo é especulativa..."*

---

### 📌 **2ª Execução — Resposta Crítica (Alucinação / Confiança Excessiva)**

* **Pergunta do Usuário:**  
  > *"Quais são as principais criptos no ano de 2026"*

* **Raciocínio Interno (*Reasoning*):**  
  > *"Need to answer with up-to-date info as of 2026. Provide list, maybe top by market cap, usage, trends. Also mention emerging ones, DeFi, layer 2, etc. Should be factual..."*

* **Comportamento e Resposta:**  
  Nesta chamada, o modelo assumiu que deveria responder como se **já estivesse no ano de 2026** (setembro de 2026), sem incluir qualquer aviso sobre o limite do seu treinamento.

  > **Trecho da Resposta:** *"## Principais criptomoedas em 2026 \n\nA lista abaixo reúne as criptomoedas que, até setembro 2026, se destacam..."*

---

### 💡 **Conclusão & Observações**

1. **Inconsistência de Segurança:** Rodando exatamente o mesmo prompt com `temperature > 0`, o modelo oscilou entre **reconhecer suas limitações temporais** (Execução 1) e **gerar dados ficcionais/projetados com tom factual** (Execução 2).
2. **Impacto Prático:** A 2ª resposta é crítica para aplicações em produção, pois apresenta estimativas como fatos históricos reais ("até setembro de 2026"), o que pode induzir o usuário final ao erro.
3. **Ação Recomendada:** Para evitar essa oscilação, é recomendável ajustar o `System Prompt` para instruir explicitamente o modelo a recusar respostas factuais sobre datas futuras ou sempre exigir a inclusão de um *disclaimer* de limitação temporal.

In [11]:
# System Prompt focado em reflexão/raciocínio prévio sobre limitações temporais
system_template = (
    "Você é um assistente de IA especialista, altamente crítico e transparente.\n\n"
    "Antes de responder à solicitação do usuário, faça uma verificação interna rigorosa:\n"
    "1. Avalie se o pedido exige informações, estatísticas ou eventos em tempo real, "
    "ou referentes a um ano/período do qual você NÃO possui dados factuais consolidados.\n"
    "2. Se você constatar que a pergunta trata sobre o ano {ano_alvo} ou qualquer período "
    "que extrapole seus dados reais, você DEVE incluir obrigatoriamente um aviso no início da resposta.\n"
    "3. Deixe claro no aviso que a análise a seguir é **estritamente especulativa/projetada** "
    "com base em tendências históricas e que você não possui acesso a dados reais do futuro.\n"
    "4. NUNCA trate projeções hipotéticas como se fossem fatos confirmados."
)

human_template = "{input}"

# Construindo o ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_template),
    HumanMessagePromptTemplate.from_template(human_template),
])

In [12]:
chain = prompt_template | modelo

# Invocando com os parâmetros
resposta = chain.invoke({
    "ano_alvo": "2026",
    "input": "Quais são as principais criptos no ano de 2026?"
})

print(resposta.content)

**Aviso:** A resposta a seguir é **estritamente especulativa e projetada** com base nas tendências históricas e nas informações disponíveis até o final de 2024. Não possuo acesso a dados reais ou estatísticas do ano de 2026, portanto tudo o que segue deve ser interpretado como uma hipótese, não como fato confirmado.  

---

### Possíveis principais criptomoedas em 2026 (hipótese)

| Posição (hipotética) | Criptomoeda | Motivo da projeção |
|----------------------|-------------|--------------------|
| 1️⃣ | **Bitcoin (BTC)** | Continuação da reputação de “reserva de valor” e da maior capitalização de mercado. A adoção institucional e a presença em carteiras de investimento ainda são fortes impulsionadores. |
| 2️⃣ | **Ethereum (ETH)** | Domínio nas plataformas de contratos inteligentes, com a rede já consolidada após a transição para proof‑of‑stake (Ethereum 2.0). O ecossistema DeFi, NFTs e Web3 ainda depende fortemente da camada base do Ethereum. |
| 3️⃣ | **Binance Coin (BNB)** | A Bi